# Model Selection

The assessment asks to forecast hourly yellow taxi demand by NYC taxi zone. Because the target is `trip_count`, this is a **regression / forecasting** problem with a numeric target.

Main model-selection metrics:

- **MAE**: average absolute error in trips
- **RMSE**: penalizes large misses more heavily
- **WAPE**: total absolute error divided by total actual demand, common in demand forecasting
- **R2**: how much variance the model explains

## 1. Import libraries and define paths

This notebook uses models such as Decision Tree, Random Forest, XGBoost, LightGBM, and CatBoost. 

In [19]:
from pathlib import Path
import importlib.util
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


def find_project_dir(start_path=None):
    current = Path.cwd() if start_path is None else Path(start_path).resolve()
    for candidate in [current, *current.parents]:
        prepared = candidate / "data" / "processed" / "hourly_taxi_demand_2023_prepared.parquet"
        if prepared.exists():
            return candidate
    raise FileNotFoundError("Could not find data/processed/hourly_taxi_demand_2023_prepared.parquet")


PROJECT_DIR = find_project_dir()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PREPARED_PATH = PROCESSED_DIR / "hourly_taxi_demand_2023_prepared.parquet"
MODEL_RESULTS_PATH = PROCESSED_DIR / "model_selection_results.csv"

print(f"Project directory: {PROJECT_DIR}")
print(f"Prepared data: {PREPARED_PATH}")

Project directory: c:\Users\USER\Downloads\calibrax\forecasting-taxi-demand-across-nyc-zones
Prepared data: c:\Users\USER\Downloads\calibrax\forecasting-taxi-demand-across-nyc-zones\data\processed\hourly_taxi_demand_2023_prepared.parquet


## 2. Load prepared data

The prepared data already contains cleaned hourly demand, weather features, holiday features, calendar features, and lag features.

In [20]:
df = pd.read_parquet(PREPARED_PATH)

print(df.shape)
print(df["pickup_hour"].min(), "to", df["pickup_hour"].max())
print("Zones:", df["PULocationID"].nunique())
df.head()

(2303880, 51)
2023-01-01 00:00:00 to 2023-12-31 23:00:00
Zones: 263


,pickup_hour,PULocationID,trip_count,revenue,avg_trip_distance,avg_fare_amount,avg_trip_duration_min,zero_passenger_count,Borough,Zone,service_zone,weather_station_key,weather_station_name,STATION,usaf,wban,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm,has_precipitation,date,holiday_date,holiday_name,holiday_date_type,is_holiday,is_day_before_holiday,is_day_after_holiday,hour,dayofweek,dayofyear,weekofyear,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,trip_count_lag_1,trip_count_lag_2,trip_count_lag_3,trip_count_lag_24,trip_count_lag_48,trip_count_lag_72,trip_count_lag_168,trip_count_rolling_24_mean,trip_count_rolling_168_mean
0,2023-01-01 00:00:00,1,0.0,0.0,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,11.700000,10.6,1007.2,3.1,4828.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,0,6,1,52,1,1,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-01 01:00:00,1,0.0,0.0,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,11.133333,10.0,1007.2,3.1,6964.666667,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,1,6,1,52,1,1,0.258819,0.965926,-0.781831,0.62349,0.5,0.866025,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-01 02:00:00,1,0.0,0.0,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,9.400000,8.3,1008.1,2.6,6437.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,2,6,1,52,1,1,0.500000,0.866025,-0.781831,0.62349,0.5,0.866025,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-01 03:00:00,1,0.0,0.0,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,8.900000,7.8,1008.4,2.1,8047.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,3,6,1,52,1,1,0.707107,0.707107,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-01 04:00:00,1,0.0,0.0,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,9.400000,8.3,1008.9,3.6,8047.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,4,6,1,52,1,1,0.866025,0.500000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Choose target and features

Target:

- `trip_count`: number of yellow taxi pickups in a zone during an hour

Important leakage rule:

I do not use same-hour fields that would only be known after the trip happens, such as same-hour revenue, average fare, average trip distance, or average trip duration. 

i use:

- zone ID and zone metadata
- weather features
- holiday features
- calendar features
- lagged and rolling historical demand

In [21]:
target = "trip_count"

numeric_features = [
    "PULocationID",
    "temp_c",
    "dew_c",
    "pressure_hpa",
    "wind_speed_mps",
    "visibility_m",
    "precip_mm",
    "has_precipitation",
    "is_holiday",
    "is_day_before_holiday",
    "is_day_after_holiday",
    "hour",
    "dayofweek",
    "dayofyear",
    "weekofyear",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "trip_count_lag_1",
    "trip_count_lag_2",
    "trip_count_lag_3",
    "trip_count_lag_24",
    "trip_count_lag_48",
    "trip_count_lag_72",
    "trip_count_lag_168",
    "trip_count_rolling_24_mean",
    "trip_count_rolling_168_mean",
]

categorical_features = [
    "Borough",
    "service_zone",
    "weather_station_key",
    "holiday_name",
]

model_data = df[["pickup_hour", target] + numeric_features + categorical_features].copy()

# Lag features are null at the start of each zone's time series.
# I drop those early rows because the models need historical demand inputs.
required_lag_cols = ["trip_count_lag_168", "trip_count_rolling_24_mean", "trip_count_rolling_168_mean"]
model_data = model_data.dropna(subset=required_lag_cols).reset_index(drop=True)

print(model_data.shape)
model_data.head()

(2259696, 38)


,pickup_hour,trip_count,PULocationID,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm,has_precipitation,is_holiday,is_day_before_holiday,is_day_after_holiday,hour,dayofweek,dayofyear,weekofyear,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,trip_count_lag_1,trip_count_lag_2,trip_count_lag_3,trip_count_lag_24,trip_count_lag_48,trip_count_lag_72,trip_count_lag_168,trip_count_rolling_24_mean,trip_count_rolling_168_mean,Borough,service_zone,weather_station_key,holiday_name
0,2023-01-08 00:00:00,0.0,1,3.30,-3.9,1024.5,2.60,16093.0,0.0,0,0,0,0,0,6,8,1,1,1,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
1,2023-01-08 01:00:00,0.0,1,3.05,-3.9,1024.6,2.35,16046.5,0.0,0,0,0,0,1,6,8,1,1,1,0.258819,0.965926,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
2,2023-01-08 02:00:00,0.0,1,2.20,-3.9,1025.4,3.10,16093.0,0.0,0,0,0,0,2,6,8,1,1,1,0.500000,0.866025,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
3,2023-01-08 03:00:00,0.0,1,2.20,-3.9,1025.6,3.60,16093.0,0.0,0,0,0,0,3,6,8,1,1,1,0.707107,0.707107,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
4,2023-01-08 04:00:00,0.0,1,2.20,-5.6,1026.2,3.60,16093.0,0.0,0,0,0,0,4,6,8,1,1,1,0.866025,0.500000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.25,0.738095,EWR,EWR,newark,None


## 4. Time-based train / validation / test split

The model train on the past and predict the future.

- Train: January to October 2023
- Validation: November 2023, used for model selection
- Test: December 2023, reserved for final evaluation after model selection

In [22]:
train_mask = model_data["pickup_hour"] < "2023-11-01"
valid_mask = (model_data["pickup_hour"] >= "2023-11-01") & (model_data["pickup_hour"] < "2023-12-01")
test_mask = model_data["pickup_hour"] >= "2023-12-01"

train_df = model_data.loc[train_mask].copy()
valid_df = model_data.loc[valid_mask].copy()
test_df = model_data.loc[test_mask].copy()

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(valid_df), len(test_df)],
        "start": [train_df["pickup_hour"].min(), valid_df["pickup_hour"].min(), test_df["pickup_hour"].min()],
        "end": [train_df["pickup_hour"].max(), valid_df["pickup_hour"].max(), test_df["pickup_hour"].max()],
        "total_trips": [train_df[target].sum(), valid_df[target].sum(), test_df[target].sum()],
    }
)

split_summary

,split,rows,start,end,total_trips
0,train,1874664,2023-01-08,2023-10-31 23:00:00,30112143.0
1,validation,189360,2023-11-01,2023-11-30 23:00:00,3249632.0
2,test,195672,2023-12-01,2023-12-31 23:00:00,3292743.0


## 5. Encode categorical features

Tree models need numeric inputs. I use one-hot encoding for small categorical columns such as borough, service zone, weather station, and holiday name.

`PULocationID` remains numeric. For tree-based models, this is acceptable because trees can learn different splits for different zone IDs.

In [ ]:
feature_cols = numeric_features + categorical_features

X_all = pd.get_dummies(
    model_data[feature_cols],
    columns=categorical_features,
    dummy_na=False,
    dtype=np.int8,
)

# LightGBM does not allow special JSON characters in feature names.
# One-hot encoding can create names such as holiday_name_New Year's Day,
# so I sanitize all feature names once and use the same names for every model.
def clean_feature_names(columns):
    cleaned = (
        pd.Series(columns)
        .str.replace(r"[^A-Za-z0-9_]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    counts = {}
    unique_names = []
    for name in cleaned:
        if name not in counts:
            counts[name] = 0
            unique_names.append(name)
        else:
            counts[name] += 1
            unique_names.append(f"{name}_{counts[name]}")

    return unique_names


X_all.columns = clean_feature_names(X_all.columns)

X_train_full = X_all.loc[train_mask].copy()
X_valid = X_all.loc[valid_mask].copy()
X_test = X_all.loc[test_mask].copy()

y_train_full = train_df[target].astype(float)
y_valid = valid_df[target].astype(float)
y_test = test_df[target].astype(float)

print(X_train_full.shape, X_valid.shape, X_test.shape)
X_train_full.head()

(1874664, 61) (189360, 61) (195672, 61)


,PULocationID,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm,has_precipitation,is_holiday,is_day_before_holiday,is_day_after_holiday,hour,dayofweek,dayofyear,weekofyear,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,trip_count_lag_1,trip_count_lag_2,trip_count_lag_3,trip_count_lag_24,trip_count_lag_48,trip_count_lag_72,trip_count_lag_168,trip_count_rolling_24_mean,trip_count_rolling_168_mean,Borough_Bronx,Borough_Brooklyn,Borough_EWR,Borough_Manhattan,Borough_Queens,Borough_Staten_Island,Borough_nan,service_zone_Airports,service_zone_Boro_Zone,service_zone_EWR,service_zone_Yellow_Zone,service_zone_nan,weather_station_key_central_park,weather_station_key_jfk,weather_station_key_laguardia,weather_station_key_newark,weather_station_key_nan,holiday_name_Birthday_of_Martin_Luther_King_Jr,holiday_name_Christmas_Day,holiday_name_Columbus_Day,holiday_name_Independence_Day,holiday_name_Juneteenth_National_Independence_Day,holiday_name_Labor_Day,holiday_name_Memorial_Day,holiday_name_None,holiday_name_Thanksgiving_Day,holiday_name_Veterans_Day,holiday_name_Washington_s_Birthday,holiday_name_nan
0,1,3.30,-3.9,1024.5,2.60,16093.0,0.0,0,0,0,0,0,6,8,1,1,1,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
1,1,3.05,-3.9,1024.6,2.35,16046.5,0.0,0,0,0,0,1,6,8,1,1,1,0.258819,0.965926,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
2,1,2.20,-3.9,1025.4,3.10,16093.0,0.0,0,0,0,0,2,6,8,1,1,1,0.500000,0.866025,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,1,2.20,-3.9,1025.6,3.60,16093.0,0.0,0,0,0,0,3,6,8,1,1,1,0.707107,0.707107,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,1,2.20,-5.6,1026.2,3.60,16093.0,0.0,0,0,0,0,4,6,8,1,1,1,0.866025,0.500000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0


## 6. smaller training sample for faster model selection

Random Forest, XGBoost, LightGBM, and CatBoost can train on the full table, but model selection can be slow with more than 1.9 million training rows. Thus, I compare models on a reproducible training sample.

The validation set remains the full November period. This keeps the comparison fair.

set `TRAIN_SAMPLE_ROWS = None` to train on the full training set.

In [24]:
RANDOM_STATE = 42
TRAIN_SAMPLE_ROWS = 400_000

# Sampling keeps model comparison practical while preserving the full validation period.
if TRAIN_SAMPLE_ROWS is not None and len(X_train_full) > TRAIN_SAMPLE_ROWS:
    sample_index = train_df.sample(n=TRAIN_SAMPLE_ROWS, random_state=RANDOM_STATE).index
    X_train = X_train_full.loc[sample_index].copy()
    y_train = y_train_full.loc[sample_index].copy()
else:
    X_train = X_train_full.copy()
    y_train = y_train_full.copy()

print("Training rows used for model selection:", len(X_train))
print("Validation rows:", len(X_valid))

Training rows used for model selection: 400000
Validation rows: 189360


## 7. Define evaluation metrics

For demand forecasting, WAPE is especially useful because it answers: "How large is total absolute error compared with total true demand?"

Lower MAE, RMSE, and WAPE are better. Higher R2 is better.

In [25]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def wape(y_true, y_pred):
    denominator = np.sum(np.abs(y_true))
    if denominator == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denominator


def evaluate_regression(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "WAPE": wape(np.asarray(y_true), y_pred),
        "R2": r2_score(y_true, y_pred),
    }


## 8. Baseline models

the ML models will be compared against simple baselines.

- `lag_24`: predict this hour using yesterday's same hour
- `lag_168`: predict this hour using last week's same hour
- `average_lag_24_168`: average of yesterday and last week

If a complex model cannot beat these baselines, it is not useful.

In [26]:
results = []
validation_predictions = {}

baseline_predictions = {
    "Baseline_lag_24": valid_df["trip_count_lag_24"].values,
    "Baseline_lag_168": valid_df["trip_count_lag_168"].values,
    "Baseline_avg_lag_24_168": ((valid_df["trip_count_lag_24"] + valid_df["trip_count_lag_168"]) / 2).values,
}

for name, pred in baseline_predictions.items():
    # Baseline predictions are already in the original trip-count scale.
    metrics = evaluate_regression(y_valid, pred)
    metrics.update({"Model": name, "TrainSeconds": 0.0})
    results.append(metrics)
    validation_predictions[name] = np.clip(pred, 0, None)

pd.DataFrame(results).sort_values("WAPE")

,MAE,RMSE,WAPE,R2,Model,TrainSeconds
2,3.996541,14.693153,0.232883,0.926245,Baseline_avg_lag_24_168,0.0
1,4.541302,17.807640,0.264627,0.891663,Baseline_lag_168,0.0
0,4.929726,19.196597,0.287261,0.874104,Baseline_lag_24,0.0


## 9. Candidate ML models

The target is skewed: many zone-hours have low demand, while busy Manhattan and airport zones can have high demand. To make models more stable, each ML model is trained on `log1p(trip_count)` and predictions are converted back using `expm1`.

Models included:

- **Decision Tree Regressor**: easy to explain, but can overfit
- **Random Forest Regressor**: many decision trees averaged together, usually more stable than one tree
- **XGBoost Regressor**: boosted trees, strong tabular model
- **LightGBM Regressor**: boosted trees optimized for speed and large tabular data
- **CatBoost Regressor**: boosted trees that often perform well with categorical/tabular data

In [27]:
def library_available(name):
    return importlib.util.find_spec(name) is not None


models = []

models.append(
    (
        "DecisionTreeRegressor",
        DecisionTreeRegressor(
            max_depth=24,
            min_samples_leaf=25,
            random_state=RANDOM_STATE,
        ),
    )
)

models.append(
    (
        "RandomForestRegressor",
        RandomForestRegressor(
            n_estimators=80,
            max_depth=24,
            min_samples_leaf=10,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    )
)

if library_available("xgboost"):
    from xgboost import XGBRegressor

    models.append(
        (
            "XGBRegressor",
            XGBRegressor(
                n_estimators=500,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        )
    )

if library_available("lightgbm"):
    from lightgbm import LGBMRegressor

    models.append(
        (
            "LGBMRegressor",
            LGBMRegressor(
                n_estimators=600,
                learning_rate=0.04,
                num_leaves=96,
                max_depth=-1,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="regression",
                n_jobs=-1,
                random_state=RANDOM_STATE,
                verbosity=-1,
            ),
        )
    )

if library_available("catboost"):
    from catboost import CatBoostRegressor

    models.append(
        (
            "CatBoostRegressor",
            CatBoostRegressor(
                iterations=600,
                depth=8,
                learning_rate=0.05,
                loss_function="RMSE",
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
            ),
        )
    )

[name for name, _ in models]

['DecisionTreeRegressor',
 'RandomForestRegressor',
 'XGBRegressor',
 'LGBMRegressor',
 'CatBoostRegressor']

## 10. Train and compare candidate models

This cell fits each candidate model on the training data and evaluates it on November validation data.

In [28]:
# log1p reduces the effect of very high-demand zone-hours during training.
y_train_log = np.log1p(y_train)

trained_models = {}

for name, model in models:
    print(f"Training {name}...")
    start = time.time()

    model.fit(X_train, y_train_log)
    pred_log = model.predict(X_valid)
    # Convert predictions back from log scale to trip-count scale.
    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)

    elapsed = time.time() - start

    metrics = evaluate_regression(y_valid, pred)
    metrics.update({"Model": name, "TrainSeconds": elapsed})

    results.append(metrics)
    validation_predictions[name] = pred
    trained_models[name] = model

results_df = pd.DataFrame(results)
ordered_cols = [
    "Model",
    "MAE",
    "RMSE",
    "WAPE",
    "R2",
    "TrainSeconds",
]

results_df = results_df[ordered_cols].sort_values("WAPE").reset_index(drop=True)
results_df.to_csv(MODEL_RESULTS_PATH, index=False)
results_df

Training DecisionTreeRegressor...
Training RandomForestRegressor...
Training XGBRegressor...
Training LGBMRegressor...
Training CatBoostRegressor...


,Model,MAE,RMSE,WAPE,R2,TrainSeconds
0,LGBMRegressor,2.910743,10.382759,0.169613,0.963171,33.023609
1,XGBRegressor,2.978208,10.896217,0.173544,0.959439,57.058592
2,RandomForestRegressor,3.059860,11.411561,0.178302,0.955511,53.592858
3,CatBoostRegressor,3.067083,11.056339,0.178723,0.958238,35.708788
4,DecisionTreeRegressor,3.238692,11.409220,0.188723,0.955529,14.492443
5,Baseline_avg_lag_24_168,3.996541,14.693153,0.232883,0.926245,0.000000
6,Baseline_lag_168,4.541302,17.807640,0.264627,0.891663,0.000000
7,Baseline_lag_24,4.929726,19.196597,0.287261,0.874104,0.000000


## 11. Select the best model

The main selection metric is WAPE because this is a demand forecasting task. Lower WAPE means the model has lower total absolute error relative to total demand.

In [29]:
best_row = results_df.iloc[0]
best_model_name = best_row["Model"]

print("Best validation model:", best_model_name)
best_row

Best validation model: LGBMRegressor


Model           LGBMRegressor
MAE                  2.910743
RMSE                10.382759
WAPE                 0.169613
R2                   0.963171
TrainSeconds        33.023609
Name: 0, dtype: object

## 12. Error analysis by borough and hour

After selecting a model, i inspect where it performs well or poorly. This is important for the report because average metrics alone can hide weak performance in specific boroughs or time periods.

In [30]:
best_valid_pred = validation_predictions[best_model_name]

valid_error = valid_df[["pickup_hour", "PULocationID", "Borough", "hour", target]].copy()
valid_error["prediction"] = best_valid_pred
valid_error["abs_error"] = (valid_error[target] - valid_error["prediction"]).abs()

borough_error = (
    valid_error.groupby("Borough")
    .agg(
        rows=(target, "size"),
        actual_trips=(target, "sum"),
        predicted_trips=("prediction", "sum"),
        MAE=("abs_error", "mean"),
        total_abs_error=("abs_error", "sum"),
    )
    .reset_index()
)
borough_error["WAPE"] = borough_error["total_abs_error"] / borough_error["actual_trips"].replace(0, np.nan)
borough_error.sort_values("WAPE")

,Borough,rows,actual_trips,predicted_trips,MAE,total_abs_error,WAPE
3,Manhattan,49680,2919789.0,2.776758e+06,9.106704,452421.039252,0.154950
4,Queens,49680,301439.0,2.690717e+05,1.429474,71016.273292,0.235591
1,Brooklyn,43920,21997.0,1.666822e+04,0.451868,19846.060468,0.902217
2,EWR,720,374.0,2.581347e+02,0.552536,397.826063,1.063706
0,Bronx,30960,5968.0,4.152572e+03,0.238101,7371.622277,1.235191
5,Staten Island,14400,65.0,6.344923e+01,0.008713,125.471100,1.930325


In [31]:
hour_error = (
    valid_error.groupby("hour")
    .agg(
        actual_trips=(target, "sum"),
        predicted_trips=("prediction", "sum"),
        MAE=("abs_error", "mean"),
        total_abs_error=("abs_error", "sum"),
    )
    .reset_index()
)
hour_error["WAPE"] = hour_error["total_abs_error"] / hour_error["actual_trips"].replace(0, np.nan)
hour_error.sort_values("hour")

,hour,actual_trips,predicted_trips,MAE,total_abs_error,WAPE
0,0,82630.0,76469.477592,2.569559,20273.820075,0.245357
1,1,56440.0,51127.143029,1.969029,15535.638086,0.275259
2,2,35346.0,34943.413267,1.489774,11754.317103,0.332550
3,3,23991.0,22628.467123,1.019502,8043.867358,0.335287
4,4,17989.0,15747.155196,0.996074,7859.020171,0.436879
5,5,19729.0,16778.434506,1.028515,8114.981911,0.411323
6,6,43892.0,39943.240953,1.517297,11971.474318,0.272748
7,7,90120.0,79948.470607,2.685693,21190.117959,0.235132
8,8,122951.0,113718.165830,2.902903,22903.908598,0.186285
9,9,137484.0,129905.265471,2.744800,21656.472032,0.157520


## 13. final test evaluation

The December test set should be used only after model selection. If the selected model is an ML model, this cell retrains that model type using both train and validation periods, then evaluates on December.

If the selected model is a baseline, it evaluates that baseline directly on December.

In [ ]:
test_results = None

if best_model_name.startswith("Baseline"):
    if best_model_name == "Baseline_lag_24":
        test_pred = test_df["trip_count_lag_24"].values
    elif best_model_name == "Baseline_lag_168":
        test_pred = test_df["trip_count_lag_168"].values
    else:
        test_pred = ((test_df["trip_count_lag_24"] + test_df["trip_count_lag_168"]) / 2).values
else:
    # Recreate the chosen model with the same settings, then train on January to November.
    model_factory = dict(models)
    final_model = model_factory[best_model_name]

    X_train_valid_full = pd.concat([X_train_full, X_valid], axis=0)
    y_train_valid_full = pd.concat([y_train_full, y_valid], axis=0)

    # Keep final training practical. Set FINAL_TRAIN_SAMPLE_ROWS = None to use all Jan-Nov rows
    FINAL_TRAIN_SAMPLE_ROWS = 600_000
    if FINAL_TRAIN_SAMPLE_ROWS is not None and len(X_train_valid_full) > FINAL_TRAIN_SAMPLE_ROWS:
        final_sample_index = y_train_valid_full.sample(n=FINAL_TRAIN_SAMPLE_ROWS, random_state=RANDOM_STATE).index
        X_final_train = X_train_valid_full.loc[final_sample_index]
        y_final_train = y_train_valid_full.loc[final_sample_index]
    else:
        X_final_train = X_train_valid_full
        y_final_train = y_train_valid_full

    final_model.fit(X_final_train, np.log1p(y_final_train))
    test_pred = np.expm1(final_model.predict(X_test))

test_pred = np.clip(test_pred, 0, None)
test_results = evaluate_regression(y_test, test_pred)
test_results["Model"] = best_model_name

pd.DataFrame([test_results])

,MAE,RMSE,WAPE,R2,Model
0,3.650255,12.627882,0.216917,0.941784,LGBMRegressor


## 14. Recommended model-selection explanation for report

Use this wording in your report/interview:

> I treated hourly taxi demand as a regression forecasting task because the target is a numeric trip count. I used a chronological split to avoid leakage: January-October for training, November for validation/model selection, and December for final testing. I compared simple seasonal baselines against tree-based regression models that I can explain clearly: Decision Tree, Random Forest, XGBoost, LightGBM, and CatBoost. The main metric was WAPE because demand forecasting cares about total absolute error relative to total actual demand. MAE, RMSE, and R2 were also reported.

The best model should be selected primarily by validation WAPE, then checked for reasonable performance by borough, zone, and hour.